In [ ]:
# %pip install python-dotenv
# %uv add dspy

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


In [2]:
import os
# os.environ['OPENAI_API_KEY'] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [3]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=0)
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [4]:
print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [5]:
from gepa_artifact.benchmarks.super_bench import benchmark as sb_metas

/mnt/c/Users/2825425/work/gepa-research/gepa-modified/.venv/lib/python3.12/site-packages/aicodetools/tools/read.py:51: SyntaxWarning: invalid escape sequence '\w'
  """


In [6]:
bench = sb_metas[0].benchmark()

In [7]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(2, 2, 2)

In [8]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'instance_id': 'mera', 'github_repo': 'https://github.com/ai-forever/MERA', 'git_commit': '1923853c13dbc11d140eba4dbbf8386bf746e609', 'query': 'Use the lm-evaluation-harness to evaluate ai-forever/rugpt3small_based_on_gpt2 on the chegeka test set. Report "metric" and "metric_stderr" as a json structured as follows: {"metric": 0.0, "metric_stderr": 0.0} (replace 0.0 with the actual values).\n\nAdditional instructions:\n1. Load only the first 10 rows of the dataset.\n2. Use the following hyperparameters: batch_size=1, num_fewshot=4\n\nGit repository: https://github.com/ai-forever/MERA', 'query_components': {'e2e_task': 'Use the lm-evaluation-harness to evaluate ai-forever/rugpt3small_based_on_gpt2 on the chegeka test set.', 'scenario_task': '', 'report': 'Report "metric" and "metric_stderr" as a json structured as follows: {"metric": 0.0, "metric_stderr": 0.0} (replace 0.0 with the actual values).', 'instructions': '1. Load only the first 10 rows of the dataset.\n2. Use the foll

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [ ]:
program = lb_math_metas[0].program[0]
program

## Define an evaluator and evaluate the base program

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=lb_math_metas[0].metric,
    num_threads=80,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set)
)

In [ ]:
evaluate(program)

## Load the GEPA Optimizer

In [ ]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if lb_math_metas[0].feedback_fn_maps is None or lb_math_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = lb_math_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = lb_math_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=lb_math_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    max_metric_calls=700,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=40
)

## Optimize the program with GEPA

In [ ]:
optimized_program = optimizer.compile(
    lb_math_metas[0].program[0],
    trainset=bench.train_set,
    valset=bench.val_set[:len(bench.val_set)//2],
)

## Now, let's evaluate the optimized program

In [ ]:
evaluate(optimized_program)

GEPA was able to optimize the base program **from 57% score to 61% score** in just 9 iterations. With higher budget, the optimized program's score can go as high as **64%**.

### Let's print the prompts that GEPA discovered

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")